# Tema 03: Seaborn

**Taller:** Análisis y Visualización Interactiva en Python
**Duración estimada de esta sesión:** 60 minutos
**Herramienta principal:** Seaborn (sobre Matplotlib)
**Modalidad de práctica:** Google Colab

---

> 📌 **Nota para el profesor:** esta notebook está diseñada para proyectarse y ejecutarse en vivo. Las secciones marcadas como **Práctica guiada** se resuelven junto con el grupo; las marcadas como **Práctica independiente** las resuelven los participantes en sus propias copias de la notebook (`Archivo → Guardar una copia en Drive`).

## 🎯 Objetivos de aprendizaje

Al finalizar este tema, el participante será capaz de:

- Explicar la relación entre Seaborn y Matplotlib, y cuándo conviene usar cada uno.
- Construir gráficos estadísticos (boxplot, violinplot, histplot, scatterplot con regresión) directamente desde un DataFrame.
- Usar `hue`, `col`/`row` para comparar subgrupos sin escribir bucles manuales.
- Construir un mapa de calor de correlaciones y un `pairplot` para exploración multivariada.
- Aplicar temas y paletas de color consistentes a un conjunto de gráficos.

## 🧠 Contenido teórico

### 1. ¿Qué es Seaborn?

**Seaborn** es una librería construida **sobre Matplotlib**, orientada a la visualización **estadística**. Su API está diseñada para trabajar directamente con DataFrames "ordenados" (*tidy data*: una fila por observación, una columna por variable), lo que reduce drásticamente la cantidad de código necesaria frente a Matplotlib puro.

| | Matplotlib | Seaborn |
|---|---|---|
| Nivel de abstracción | Bajo (control total, más código) | Alto (menos código, defaults estadísticos) |
| Entrada típica | Arreglos `x`, `y` | DataFrame + nombres de columnas |
| Fortaleza | Personalización pixel a pixel | Comparar grupos, distribuciones, correlaciones |
| Relación | Base | Construida sobre Matplotlib (puedes seguir usando `ax.set_title()`, etc.) |

**Regla práctica para el taller:** si la pregunta es *"¿cómo se compara la distribución de X entre grupos?"* o *"¿cómo se relacionan estas variables numéricas?"*, Seaborn casi siempre lo resuelve en una línea.

### 2. Familias de gráficos en Seaborn

- **Relacionales** (`scatterplot`, `lineplot`): relación entre dos variables numéricas, opcionalmente con líneas de tendencia (`regplot`, `lmplot`).
- **Distribución** (`histplot`, `kdeplot`, `boxplot`, `violinplot`): cómo se distribuye una variable, y cómo cambia entre categorías (`hue`).
- **Categóricos** (`barplot`, `countplot`, `stripplot`): comparaciones entre categorías con intervalos de confianza calculados automáticamente.
- **Matriciales** (`heatmap`): matrices como correlaciones.
- **Multigráfico** (`pairplot`, `FacetGrid`, `catplot` con `col=`): "pequeños múltiplos" — una cuadrícula de gráficos, uno por subgrupo, generada automáticamente.

### 3. El parámetro `hue` (y `col`/`row`)

El superpoder de Seaborn es el parámetro `hue`: colorea automáticamente por una tercera variable categórica, calculando leyenda, paleta y (cuando aplica) estadísticos por grupo sin bucles manuales. `col=` / `row=` hacen lo mismo pero generando subgráficos separados en una cuadrícula (*faceting*).

### 4. Estética y temas

`sns.set_theme(style=..., palette=...)` cambia la apariencia de **todos** los gráficos posteriores (incluso los de Matplotlib) de forma consistente — muy útil para mantener un estilo uniforme en un reporte o repositorio.

## ⚙️ Configuración del entorno

Cargaremos `tema03_salarios_tecnologia.csv`: salarios sintéticos del sector tecnológico por puesto, experiencia, país, género, nivel educativo y modalidad de trabajo.

In [ ]:
# === Carga del dataset ===
# Opción 1 (recomendada una vez publicado el repositorio del taller):
# reemplaza <usuario>/<repositorio> por la ruta real de tu repo de GitHub
# y ejecuta esta celda. Usa el botón "Raw" de GitHub para obtener la URL.
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/<usuario>/<repositorio>/main/"
    "datasets/tema03_salarios_tecnologia.csv"
)

import pandas as pd

try:
    df = pd.read_csv(GITHUB_RAW_URL)
    print("Datos cargados desde GitHub ✅  ->", df.shape)
except Exception as e:
    print("No se pudo leer desde GitHub todavía (repo no configurado o sin internet).")
    print("Sube manualmente el archivo 'tema03_salarios_tecnologia.csv' cuando se te solicite.")
    try:
        from google.colab import files
        subido = files.upload()  # selecciona tema03_salarios_tecnologia.csv
        df = pd.read_csv(list(subido.keys())[0])
    except ImportError:
        # Fuera de Colab (por ejemplo, ejecución local de prueba):
        df = pd.read_csv("tema03_salarios_tecnologia.csv")

df.head()

## 🧭 Práctica guiada

### Paso 0 · Tema visual

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="viridis")

### Paso 1 · Boxplot — distribución de salario por puesto

In [ ]:
orden_puestos = (
    df.groupby("puesto")["salario_anual_mxn_equiv"].median().sort_values(ascending=False).index
)

plt.figure(figsize=(11, 5))
sns.boxplot(data=df, x="puesto", y="salario_anual_mxn_equiv", order=orden_puestos)
plt.title("Distribución de salario anual por puesto")
plt.xticks(rotation=35, ha="right")
plt.ylabel("Salario anual (equivalente MXN)")
plt.xlabel("")
plt.tight_layout()
plt.show()

### Paso 2 · Violinplot con `hue` — comparar además por modalidad de trabajo

In [ ]:
plt.figure(figsize=(9, 5))
sns.violinplot(
    data=df[df["puesto"].isin(["Data Scientist", "ML Engineer", "Data Analyst"])],
    x="puesto", y="salario_anual_mxn_equiv", hue="modalidad_trabajo",
    split=False,
)
plt.title("Salario por puesto y modalidad de trabajo")
plt.tight_layout()
plt.show()

### Paso 3 · Relación entre variables numéricas con regresión

In [ ]:
sns.lmplot(
    data=df, x="anos_experiencia", y="salario_anual_mxn_equiv",
    hue="nivel_educativo", height=5, aspect=1.4, scatter_kws={"alpha": 0.4},
)
plt.title("Salario vs. años de experiencia, por nivel educativo")
plt.show()

### Paso 4 · Mapa de calor de correlaciones

In [ ]:
numericas = df[["anos_experiencia", "salario_anual_mxn_equiv"]].copy()
numericas["nivel_educativo_cod"] = df["nivel_educativo"].map(
    {"Técnico": 1, "Licenciatura": 2, "Maestría": 3, "Doctorado": 4}
)

corr = numericas.corr()

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()

### Paso 5 · `catplot` con `col=` — pequeños múltiplos por país

In [ ]:
paises_top = ["México", "España", "Chile"]

g = sns.catplot(
    data=df[df["pais"].isin(paises_top)],
    x="nivel_educativo", y="salario_anual_mxn_equiv",
    col="pais", kind="box",
    order=["Técnico", "Licenciatura", "Maestría", "Doctorado"],
    height=4, aspect=0.9,
)
g.set_xticklabels(rotation=30)
g.fig.suptitle("Salario por nivel educativo, por país", y=1.05)
plt.show()

## ✍️ Práctica independiente

**Ejercicio 1.** Crea un `histplot` (o `kdeplot`) de `salario_anual_mxn_equiv` con `hue='genero'` para comparar la distribución salarial por género.

In [ ]:
# TODO: tu código aquí

**Ejercicio 2.** Usa `sns.barplot` para mostrar el salario **promedio** por `tamano_empresa`, ordenado de mayor a menor.

In [ ]:
# TODO: tu código aquí

**Ejercicio 3.** Genera un `sns.scatterplot` de `anos_experiencia` vs. `salario_anual_mxn_equiv`, coloreado (`hue`) por `puesto`, y con el tamaño de punto (`size`) según `anos_experiencia`.

In [ ]:
# TODO: tu código aquí

**Ejercicio 4 (reto).** Construye un `sns.pairplot` con las columnas `anos_experiencia` y `salario_anual_mxn_equiv`, coloreado por `modalidad_trabajo`, solo para el puesto `'Data Scientist'`.

In [ ]:
# TODO: tu código aquí

---
### ✅ Soluciones (referencia para el profesor)

In [ ]:
# Ejercicio 1
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="salario_anual_mxn_equiv", hue="genero", kde=True, element="step")
plt.title("Distribución salarial por género")
plt.show()

# Ejercicio 2
orden_tam = df.groupby("tamano_empresa")["salario_anual_mxn_equiv"].mean().sort_values(ascending=False).index
plt.figure(figsize=(7, 4))
sns.barplot(data=df, x="tamano_empresa", y="salario_anual_mxn_equiv", order=orden_tam, errorbar="sd")
plt.title("Salario promedio por tamaño de empresa")
plt.show()

# Ejercicio 3
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="anos_experiencia", y="salario_anual_mxn_equiv",
                 hue="puesto", size="anos_experiencia", alpha=0.6)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title("Experiencia vs. salario, por puesto")
plt.tight_layout()
plt.show()

# Ejercicio 4
sub = df[df["puesto"] == "Data Scientist"]
sns.pairplot(sub, vars=["anos_experiencia", "salario_anual_mxn_equiv"], hue="modalidad_trabajo")
plt.show()

## 🔎 Cierre y puente al siguiente tema

Seaborn es excelente para **explorar y comunicar estadística** rápidamente, pero sigue produciendo imágenes estáticas: el usuario no puede hacer zoom, filtrar en vivo ni pasar el mouse para ver detalles. En el **Tema 04 (Plotly)** daremos el salto a visualizaciones **interactivas**.

## 📚 Recursos adicionales

- [Documentación oficial de Seaborn](https://seaborn.pydata.org/)
- [Galería de ejemplos oficiales](https://seaborn.pydata.org/examples/index.html)
- [Tutorial oficial: Visualizing statistical relationships](https://seaborn.pydata.org/tutorial/relational.html)
- [Guía de paletas de color de Seaborn](https://seaborn.pydata.org/tutorial/color_palettes.html)